In [10]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

data = pd.read_csv("cleaned_dataset.csv")

print(len(data))


7578


## Model for trains delay on arrival

In [ ]:
data["Date"] = pd.to_datetime(data["Date"], format="%Y-%m")
data["Mois"] = data["Date"].dt.month

data["retard_moyen_mois_global"] = (
    data
    .groupby(["Gare de départ", "Gare d'arrivée", "Mois"])      #Overall historical functionnality
    ["Retard moyen de tous les trains à l'arrivée"]
    .transform("mean")
)

stations = [
    "Mois",
    "Service",
    "Gare de départ",
    "Gare d'arrivée",
    "Durée moyenne du trajet",
    "Nombre de circulations prévues",
    "retard_moyen_mois_global",
]

target = "Retard moyen de tous les trains au départ"

X = pd.get_dummies(data[stations])      # Variables 'stations' encoding so that the model considers it as a number
y = data[target]

train_columns = X.columns

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestRegressor(
    n_estimators=400,
    max_depth=None,
    min_samples_split=5,                # Training of the model
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("R² :", r2_score(y_test, y_pred))
print("MAE :", mean_absolute_error(y_test, y_pred))
print("RMSE :", np.sqrt(mean_squared_error(y_test, y_pred)))

def prediction_retard(departure_station: str,
                      arrival_station: str,
                      month: int):

    d = data[
        (data["Gare de départ"] == departure_station) &
        (data["Gare d'arrivée"] == arrival_station) &
        (data["Mois"] == month)
    ]

    if d.empty:
        print("Journey not found for that date")
        return None

    number_journeys = d["Nombre de circulations prévues"].mean()
    average_duration = d["Durée moyenne du trajet"].mean()
    average_delay_month_global = d["retard_moyen_mois_global"].iloc[0]
    service = d["Service"].mode()[0]

    input_df = pd.DataFrame([[
        month,
        service,
        departure_station,
        arrival_station,
        number_journeys,
        average_duration,
        average_delay_month_global
    ]], columns=stations)

    input_encoded = pd.get_dummies(input_df)        # Station encoding

    input_encoded = input_encoded.reindex(columns=train_columns, fill_value=0)      # Station selected = 1, station not selected = 0

    prediction = model.predict(input_encoded)

    return prediction[0]

prediction_retard(
    "TOURCOING",
    "BORDEAUX ST JEAN",
    month=3
)

R² : 0.16021591723976025
MAE : 1.694802887654636
RMSE : 4.56443514825348


np.float64(5.163805249275612)

## Model for trains delay on departure

In [ ]:
data["Date"] = pd.to_datetime(data["Date"], format="%Y-%m")
data["Mois"] = data["Date"].dt.month

data["retard_moyen_mois_global"] = (
    data
    .groupby(["Gare de départ", "Gare d'arrivée", "Mois"])      #Overall historical functionnality
    ["Retard moyen de tous les trains au départ"]
    .transform("mean")
)

stations = [
    "Mois",
    "Service",
    "Gare de départ",
    "Gare d'arrivée",
    "Nombre de circulations prévues",
    "retard_moyen_mois_global",
]

target = "Retard moyen de tous les trains au départ"

X = pd.get_dummies(data[stations])      # Variables 'stations' encoding so that the model considers it as a number
y = data[target]

train_columns = X.columns

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestRegressor(
    n_estimators=400,
    max_depth=None,
    min_samples_split=5,                # Training of the model
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("R² :", r2_score(y_test, y_pred))
print("MAE :", mean_absolute_error(y_test, y_pred))
print("RMSE :", np.sqrt(mean_squared_error(y_test, y_pred)))

def prediction_retard(departure_station: str,
                      arrival_station: str,
                      month: int):

    d = data[
        (data["Gare de départ"] == departure_station) &
        (data["Gare d'arrivée"] == arrival_station) &
        (data["Mois"] == month)
    ]

    if d.empty:
        print("Journey not found for that date")
        return None

    number_journeys = d["Nombre de circulations prévues"].mean()
    average_delay_month_global = d["retard_moyen_mois_global"].iloc[0]
    service = d["Service"].mode()[0]

    input_df = pd.DataFrame([[
        month,
        service,
        departure_station,
        arrival_station,
        number_journeys,
        average_delay_month_global
    ]], columns=stations)

    input_encoded = pd.get_dummies(input_df)         # Station encoding

    input_encoded = input_encoded.reindex(columns=train_columns, fill_value=0)      # Station selected = 1, station not selected = 0

    prediction = model.predict(input_encoded)

    return prediction[0]

prediction_retard(
    "TOURCOING",
    "BORDEAUX ST JEAN",
    month=12
)

R² : 0.3080722005289497
MAE : 1.6327431480759402
RMSE : 4.143178185217449


np.float64(2.0604431635845777)